In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

In [2]:
embed_dic = {
    'MLP':'EMBED_MLP',
    'GCN2':'EMBED_GCN',
    'SGC2':'EMBED_SGC',
    'FastGCN':'EMBED_FGCN',
    'sga':'sga'
}
atked_dic = {
    'GCN':'ATKED_GCN',
    'RobustGCN':'ATKED_RGCN',
    'GCN_Jaccard':'ATKED_JGCN',
    'SimPGCN':'ATKED_SIMPGCN'
}
we_lr_dic = {
    '5e-5':'5e-5',
    '5e-05':'5e-5',
    '0.05':'5e-2',
    '0.005':'5e-3',
    '0.0005':'5e-4',
    '0.1':'1e-1',
    '0.01':'1e-2',
    '0.001':'1e-3',
    '0.0001':'1e-4'
}
datasets = ['cora', 'citeseer', 'citeseer_full', 'flickr',
                    'cora_full', 'pubmed', 'coauthor_cs', 'coauthor_phy']
added_cols = ['method','atked_model','lay_act','lay_act_cnt','hids_nums','weight_decay','lr']
cols = ['eva_asr','poi_asr','cost','embed_acc','clean_acc']
cols = ['eva_asr','poi_asr','clean_acc']
def getSplitsC(df_idx):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if "Jaccard" not in split:
            splits.append(split)
        else:
            splits.append(split[:1] + ['_'.join(split[1:3])] + split[3:])
    return splits
def getSplits(df_idx,cnt=2):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if 'sga' in split:
            blank = [''] * cnt 
            if "Jaccard" in split:
                splits.append(['sga'] + ['_'.join(split[1:3])] + blank)
            else:
                splits.append(split + blank)
        elif "Jaccard" not in split:
            splits.append([split[0], split[-1]] + split[1:-1])
        else:
            splits.append([split[0]] + ['_'.join(split[-2:])] + split[1:-2])
    return splits
def print_d(df,cnt=2):
    atked_groups = df.groupby('method')
    for embed_model, group_eles in atked_groups:
        g2 = group_eles.groupby('atked_model')
        sga_ = {}
        for atked_model, g2_eles in g2:
            sga_[atked_model] = 0
        for atked_model, g2_eles in g2:
            best = g2_eles.sort_values(by='poi_asr',ascending=False)[:1].values[0]
            if embed_model == 'sga':
                sga_[atked_model] = best[1]
                continue
    print(sga_)
    for embed_model, group_eles in atked_groups:
        g2 = group_eles.groupby('atked_model')
        print('################### start')
        for atked_model, g2_eles in g2:
            best = g2_eles.sort_values(by='poi_asr',ascending=False)[:1].values[0]
            if embed_model == 'sga':
#                 print('poi_asr:',best[1])
                continue
            s00 = best[7]
            s0 = best[8]
#             s1 = 'HIDS[' + str(int(best[9]) - cnt) + ']'
#             s2 = we_lr_dic[best[10]]
#             s3 = we_lr_dic[best[11]]
            print(embed_dic[embed_model], end=' ')
            print(atked_dic[atked_model], end=' ')
            print('['+ ', '.join([s00,s0])+']', ' poi_asr:',best[1], '>=' if float(best[1]) >= float(sga_[atked_model]) else '<', sga_[atked_model])
            print()
        print('################## end')
        print()
def getSplitsD(df_idx, _type='normal'):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if _type == 'normal':
            if "Jaccard" not in split:
                splits.append(split)
            else:
                splits.append(split[:1] + split[1:-2] + ['_'.join(split[-2:])])
        elif _type == 'tt':
            splits.append(split[:1] + ['_'.join(split[1:])])
    return splits
def save_test(_prefix, times, seeds):
    print(_prefix)
    tdf = None
    for i in range(times):
        filename = "_".join([_prefix, str(i)]) + '.csv'
        df = pd.read_csv(filename, index_col=0)
        if i == 0:
            tdf = df
        else:
            tdf += df
    tdf /= times
    tdf.index.name = ','.join(np.array(seeds).astype('str'))
    tdf.to_csv(_prefix + '_total.csv')
#     save_test('test_cluster/chameleon_2021_12_02_16_30_01', 5, [2022,2012,1997,5018,2413])
def transform(df_, embed_models, atked_models_):
    print('clean acc:')
    mape = {
        'sga':'SGA',
        'MLP':'C_MLP',
        'SGC2':'C_SGC',
        'GCN2':'C_GCN',
        'FastGCN':'C_FastGCN',
        'MLP&SGC2':'C_MLP&SGC',
        'MLP&GCN2':'C_MLP&GCN',
        'SimPGCN':'C_SimPGCN',
        'ori':'C_ORI'
    }
    embed_models_ = [mape[em] for em in embed_models]
    d = set()
    for i in range(len(df_)):
        atk_model = df_.iloc[i].name[1]
        if atk_model in d:
            continue
        d.add(atk_model)
        print('  {}:{:.3f}'.format(atk_model, df_.iloc[i].clean_acc))
    metrics_ = ['eva_asr', 'poi_asr']
    mapping = {embed_models_[i]:embed_models[i] for i in range(len(embed_models))}
    df = pd.DataFrame(index=embed_models_, columns=pd.MultiIndex.from_product([atked_models_, metrics_]))
    for em in embed_models_:
        ls = []
        for at in atked_models_:
            for asr in metrics_:
                val = 0
                if (mapping[em], at) in df_.index:
                    val = df_.loc[mapping[em]].loc[[at]][asr].values[0]
                ls.append(val)
        df.loc[em] = ls
    return df
def get_result(df, _type='normal'):
    ttm = ['SVM', 'OCSVM']
    tts = ['TEDGE', 'TBS', 'WBS', 'TBS+WBS']
    e_sort_ = ['sga', 'ori', 'MLP', 'SGC2', 'GCN2', 'FastGCN','SimPGCN', 'MLP&SGC2', 'MLP&GCN2']
    er_sort_ = [i for i in range(len(e_sort_))]
    a_sort_ = ['GCN', 'GCN_Jaccard', 'RobustGCN', 'SimPGCN', 'FAGCN', 'H2GCN2', 'H2GCN1', 'SGCPD', 'BMGCN'] + [ml + '_' + tt for ml in ttm for tt in tts]
    ar_sort_ = [i for i in range(len(a_sort_))]
    _split = []
    for idx in df.index:
        split = idx.split('_')
        tmp = '_'.join(split[1:])
        _split.append(tmp)
    atked_models_ = list(set(_split))
    atked_models_ = sorted(atked_models_, key=lambda x:ar_sort_[a_sort_.index(x)])
    embed_models = list(set([idx.split('_')[0] for idx in df.index]))
    embed_models = sorted(embed_models, key=lambda x:er_sort_[e_sort_.index(x)])
    df = df[cols].sort_values('poi_asr', ascending=False)
    df[['embed_model','atked_model']] = getSplitsD(df.index, _type)
    df = df.set_index(['embed_model','atked_model'])
    df = transform(df, embed_models, atked_models_)
    return df

In [3]:
# save_test('test_cluster/ogbn-arxiv_2021_12_14_20_39_38', 5, [2022,2012,1997,5018,2413])

# 异质数据集

## direct

In [3]:
df = pd.read_csv('chameleon_2022_01_26_17_30_04_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  SimPGCN:0.399
  FAGCN:0.501
  H2GCN1:0.389
  H2GCN2:0.401


D:\anaconda\envs\tt\lib\site-packages\IPython\core\interactiveshell.py:2895: PerformanceWarning: indexing past lexsort depth may impact performance.
  return runner(coro)


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA          0.61   0.628   0.478   0.548   0.304   0.416   0.198   0.196   
C_MLP       0.542   0.626   0.556   0.622   0.252    0.41    0.13   0.188   
C_SGC       0.752   0.774     0.6   0.624   0.272   0.352   0.172   0.206   
C_GCN        0.78   0.814   0.606   0.666    0.27    0.38    0.18    0.21   
C_FastGCN    0.72    0.75   0.656   0.692   0.324   0.418   0.162     0.2   
C_SimPGCN   0.614   0.638   0.612   0.612   0.192   0.332   0.114    0.15   
C_MLP&SGC    0.74   0.748   0.662    0.65    0.26     0.4   0.166   0.194   
C_MLP&GCN   0.592   0.674   0.586   0.634   0.258   0.382   0.146   0.184   

           H2GCN1          
          eva_asr poi_asr  
SGA         0.234   0.238  
C_MLP        0.12   0.168  
C_SGC       0.156   0.182  
C_GCN       0.168   0.188  
C_FastGCN   0.162   0.184  
C_SimPGCN   0.102   0.124  
C_MLP&SGC   0.156   0.172  
C_MLP&GCN   0.152   0.182

In [13]:
df = pd.read_csv('chameleon_2022_02_23_21_22_47_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  SimPGCN:0.399
  FAGCN:0.501
  H2GCN1:0.389
  H2GCN2:0.401


GCN         SimPGCN           FAGCN          H2GCN2          H2GCN1  \
      eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr   
C_ORI   0.574    0.62   0.492   0.516   0.278   0.386     0.2   0.206   0.214   

               
      poi_asr  
C_ORI   0.224

In [14]:
df = pd.read_csv('chameleon_2022_02_23_21_25_49_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  SimPGCN:0.399
  FAGCN:0.501
  H2GCN2:0.401
  H2GCN1:0.389


GCN         SimPGCN           FAGCN          H2GCN2          H2GCN1  \
      eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr   
C_ORI   0.456   0.486   0.394    0.42   0.194    0.25   0.144    0.15   0.114   

               
      poi_asr  
C_ORI   0.118

In [10]:
df = pd.read_csv('squirrel_2022_01_26_17_30_04_total.csv', index_col=0)
get_result(df)

clean acc:
  SimPGCN:0.271
  GCN:0.374
  FAGCN:0.310
  H2GCN2:0.258
  H2GCN1:0.246


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA         0.554   0.558   0.532   0.636   0.394   0.524   0.124   0.128   
C_MLP       0.572   0.604   0.584   0.674   0.356   0.432    0.12   0.136   
C_SGC       0.552   0.562   0.594    0.65   0.304   0.368   0.102   0.108   
C_GCN        0.57   0.578   0.616   0.668    0.35   0.416   0.112   0.114   
C_FastGCN   0.524   0.566   0.436   0.546   0.296   0.326   0.118   0.142   
C_MLP&SGC   0.538   0.572   0.602   0.668   0.356   0.422    0.12   0.124   
C_MLP&GCN   0.576   0.594    0.62   0.664   0.364    0.42   0.116   0.122   

           H2GCN1          
          eva_asr poi_asr  
SGA         0.112   0.102  
C_MLP        0.09   0.098  
C_SGC       0.078   0.082  
C_GCN       0.074    0.08  
C_FastGCN   0.082   0.088  
C_MLP&SGC   0.086   0.092  
C_MLP&GCN   0.078   0.084

In [7]:
df = pd.read_csv('film_2022_01_26_17_30_04_total.csv', index_col=0)
get_result(df)

clean acc:
  SimPGCN:0.287
  GCN:0.273
  FAGCN:0.323
  H2GCN1:0.315
  H2GCN2:0.305


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA         0.432   0.468   0.422   0.442    0.23   0.308   0.032    0.07   
C_MLP       0.396   0.408   0.526   0.514   0.242   0.274   0.026   0.056   
C_SGC       0.288     0.3   0.284   0.292   0.116   0.168   0.018   0.042   
C_GCN       0.352   0.384   0.386    0.38   0.146   0.178   0.028   0.044   
C_FastGCN   0.338   0.354   0.362    0.37   0.158   0.176   0.028   0.036   
C_MLP&SGC   0.408   0.426   0.516     0.5   0.256   0.278   0.022    0.05   
C_MLP&GCN   0.396    0.41    0.53   0.518   0.244   0.266    0.03    0.06   

           H2GCN1          
          eva_asr poi_asr  
SGA          0.09   0.112  
C_MLP        0.08   0.092  
C_SGC       0.078    0.08  
C_GCN        0.08   0.092  
C_FastGCN   0.092   0.102  
C_MLP&SGC   0.084   0.098  
C_MLP&GCN   0.078    0.09

## indirect

In [6]:
df = pd.read_csv('chameleon_2022_01_26_17_34_32_total.csv', index_col=0)
get_result(df)

clean acc:
  FAGCN:0.501
  GCN:0.552
  SimPGCN:0.399
  H2GCN2:0.401
  H2GCN1:0.389


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA         0.114   0.136   0.036   0.104    0.05   0.126   0.008   0.018   
C_MLP        0.16    0.18   0.044   0.144   0.042    0.18   0.012   0.044   
C_SGC       0.214    0.16   0.048     0.1   0.058   0.112   0.008   0.016   
C_GCN        0.19   0.154    0.04   0.102   0.044   0.128   0.008   0.012   
C_FastGCN   0.186   0.146   0.048   0.108   0.044   0.136    0.01   0.012   

           H2GCN1          
          eva_asr poi_asr  
SGA          0.01   0.012  
C_MLP       0.008    0.02  
C_SGC       0.006    0.01  
C_GCN       0.006    0.01  
C_FastGCN   0.012    0.01

In [9]:
df = pd.read_csv('squirrel_2022_01_26_17_34_32_total.csv', index_col=0)
get_result(df)

clean acc:
  SimPGCN:0.271
  GCN:0.374
  FAGCN:0.310
  H2GCN2:0.258
  H2GCN1:0.246


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA         0.268     0.3   0.028   0.296    0.05   0.178   0.002   0.012   
C_MLP       0.338   0.268   0.072    0.33   0.062   0.106    0.01   0.028   
C_SGC       0.316   0.286   0.066   0.324   0.042   0.108   0.004    0.01   
C_GCN       0.332    0.31   0.068   0.288   0.058   0.114   0.008   0.014   
C_FastGCN   0.264   0.236    0.04   0.302   0.042   0.106    0.01   0.016   

           H2GCN1          
          eva_asr poi_asr  
SGA         0.004    0.01  
C_MLP       0.006   0.014  
C_SGC       0.008   0.012  
C_GCN       0.008   0.012  
C_FastGCN   0.012   0.014

In [8]:
df = pd.read_csv('film_2022_01_26_17_34_32_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.269
  SimPGCN:0.287
  FAGCN:0.323
  H2GCN2:0.305
  H2GCN1:0.315


GCN         SimPGCN           FAGCN          H2GCN2          \
          eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr   
SGA         0.272    0.32   0.132    0.16   0.116   0.182   0.002   0.058   
C_MLP       0.304   0.358   0.188   0.194   0.112    0.17   0.002    0.05   
C_SGC        0.21   0.268    0.08    0.11    0.05   0.122   0.002    0.04   
C_GCN       0.244   0.292    0.15   0.152   0.076   0.138    0.01   0.034   
C_FastGCN   0.248    0.28   0.104   0.112   0.078   0.134   0.002   0.028   

           H2GCN1          
          eva_asr poi_asr  
SGA         0.014   0.028  
C_MLP       0.014   0.036  
C_SGC       0.012    0.02  
C_GCN       0.014    0.02  
C_FastGCN    0.01   0.022

# tedge、trans2vec

## direct

In [44]:
df = pd.read_csv('tedge_2022_01_22_01_21_23_total.csv', index_col=0)
get_result(df, 'tt')

clean acc:
  SVM_WBS:0.809
  SVM_TBS+WBS:0.814
  SVM_TBS:0.791
  SVM_TEDGE:0.800


SVM_TEDGE         SVM_TBS         SVM_WBS         SVM_TBS+WBS  \
            eva_asr poi_asr eva_asr poi_asr eva_asr poi_asr     eva_asr   
SGA            0.53   0.094   0.578   0.086   0.522    0.09       0.734   
C_MLP         0.538   0.092    0.58   0.088   0.528   0.088       0.716   
C_SGC          0.52   0.092   0.616   0.074   0.532   0.096       0.706   
C_GCN         0.508   0.094   0.602   0.096    0.53   0.104       0.682   
C_FastGCN       0.6   0.094   0.634     0.1   0.504   0.098         0.7   
C_MLP&SGC     0.546   0.096   0.608   0.086   0.566   0.104       0.674   
C_MLP&GCN      0.55   0.094    0.58   0.098   0.538    0.09       0.666   

                   
          poi_asr  
SGA         0.098  
C_MLP         0.1  
C_SGC        0.09  
C_GCN       0.096  
C_FastGCN   0.086  
C_MLP&SGC   0.086  
C_MLP&GCN   0.086

In [4]:
# ocsvm打ocsvm,svm
df = pd.read_csv('trans2vec_2022_01_26_18_03_03_total.csv', index_col=0)
get_result(df, 'tt')

clean acc:
  SVM_TBS+WBS:0.782
  OCSVM_TBS+WBS:0.863


SVM_TBS+WBS         OCSVM_TBS+WBS        
              eva_asr poi_asr       eva_asr poi_asr
SGA             0.602    0.15          0.48   0.048
C_MLP           0.612   0.136         0.468   0.034
C_SGC           0.616   0.162          0.47    0.03
C_GCN           0.592   0.174          0.48   0.038
C_FastGCN       0.652   0.146         0.484    0.04
C_MLP&SGC       0.608    0.15         0.456   0.026
C_MLP&GCN        0.63   0.142          0.47   0.034

In [63]:
# svm打svm
df = pd.read_csv('trans2vec_2022_02_02_16_19_39_total.csv', index_col=0)
get_result(df, 'tt')

clean acc:
  SVM_TBS+WBS:0.782


SVM_TBS+WBS        
              eva_asr poi_asr
SGA             0.678   0.116
C_MLP           0.644   0.116
C_SGC           0.644   0.132
C_GCN           0.628   0.112
C_FastGCN        0.65   0.118
C_MLP&SGC       0.654   0.142
C_MLP&GCN       0.616   0.122

# pd

In [66]:
# clean_acc待补充

In [3]:
df = pd.read_csv('bc1_2022_02_01_22_26_47_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.363248           0
C_MLP      0.684615           0
C_SGC      0.692877           0
C_GCN      0.583191           0
C_FastGCN  0.621368  0.00769231
C_MLP&SGC  0.745584           0
C_MLP&GCN  0.583191           0

In [7]:
df = pd.read_csv('bc2_2022_02_08_19_01_21_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.366755           0
C_MLP      0.602585           0
C_SGC      0.511454           0
C_GCN      0.558701  0.00327869
C_FastGCN  0.575182  0.00689655
C_MLP&SGC  0.566624  0.00322581
C_MLP&GCN  0.558701  0.00327869

In [8]:
df = pd.read_csv('bc3_2022_02_07_16_42_37_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.198354           0
C_MLP      0.598059   0.0106667
C_SGC      0.422517       0.006
C_GCN      0.364553   0.0105261
C_FastGCN   0.29858  0.00880272
C_MLP&SGC  0.548503   0.0106667
C_MLP&GCN  0.368934   0.0105261

In [9]:
df = pd.read_csv('bc4_2022_02_06_00_18_44_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD        
          eva_asr poi_asr
SGA          0.25   0.006
C_MLP       0.708   0.024
C_SGC       0.624   0.022
C_GCN       0.456   0.012
C_FastGCN   0.348   0.006
C_MLP&SGC     0.7   0.022
C_MLP&GCN   0.458   0.012

In [10]:
df = pd.read_csv('bc5_2022_01_29_17_31_53_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD        
          eva_asr poi_asr
SGA         0.298    0.01
C_MLP       0.528   0.016
C_SGC        0.57   0.026
C_GCN        0.41    0.02
C_FastGCN   0.472   0.022
C_MLP&SGC   0.564   0.024
C_MLP&GCN    0.41    0.02

# bmgcn

In [74]:
df = pd.read_csv('bc5_2022_01_21_20_43_00_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.672


BMGCN          
            eva_asr   poi_asr
SGA            0.49      0.35
C_MLP         0.488     0.358
C_SGC          0.33     0.362
C_GCN         0.112     0.298
C_FastGCN       0.1      0.28
C_MLP&SGC  0.382727  0.284182
C_MLP&GCN     0.164      0.19

In [5]:
df = pd.read_csv('bc10_2022_01_21_21_17_50_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.829


BMGCN        
          eva_asr poi_asr
SGA         0.342   0.288
C_MLP       0.332   0.284
C_SGC        0.27   0.252
C_GCN       0.052   0.168
C_FastGCN   0.064   0.154
C_MLP&SGC   0.222    0.27
C_MLP&GCN    0.06   0.158

In [76]:
df = pd.read_csv('bc15_2022_01_22_15_38_43_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.793


BMGCN        
          eva_asr poi_asr
SGA         0.066   0.134
C_MLP       0.142    0.19
C_SGC       0.172   0.212
C_GCN       0.046   0.142
C_FastGCN   0.042   0.124

In [77]:
df = pd.read_csv('bc20_2022_01_22_22_42_13_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.828


BMGCN        
          eva_asr poi_asr
SGA         0.152   0.176
C_MLP       0.188   0.218
C_SGC       0.214    0.25
C_GCN       0.086   0.146
C_FastGCN   0.066   0.142

In [75]:
df = pd.read_csv('bc30_2022_01_23_11_51_55_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.829


BMGCN        
          eva_asr poi_asr
SGA         0.138   0.188
C_MLP       0.192    0.22
C_SGC       0.192   0.252
C_GCN       0.118   0.168
C_FastGCN   0.116   0.174

In [4]:
df = pd.read_csv('bc50_2022_01_26_00_50_44_total.csv', index_col=0)
get_result(df)

clean acc:
  BMGCN:0.907


BMGCN        
          eva_asr poi_asr
SGA          0.05   0.062
C_MLP       0.104   0.142
C_SGC       0.236    0.21
C_GCN        0.02   0.068
C_FastGCN   0.026   0.066